In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [2]:
# ── 1. CONFIGURATION ──
CSV_PATH = '/Users/shubhbhateja/College/minor_project/NeoJaundice/chd_jaundice_published_2.csv'
IMAGE_DIR = '/Users/shubhbhateja/College/minor_project/images'


In [3]:
INPUT_SIZE = (128, 128)
SEQ_LEN = 128 * 128
# New: Metadata columns from your CSV
CLINICAL_COLS = ['gestational_age', 'age(day)', 'weight']

In [4]:
# Load the dataset
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows from CSV")

Loaded 2235 rows from CSV


In [5]:
def white_balance(img):
    """Corrects lighting using the Gray World algorithm."""
    result = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    avg_a = np.average(result[:, :, 1])
    avg_b = np.average(result[:, :, 2])
    result[:, :, 1] = result[:, :, 1] - ((avg_a - 128) * (result[:, :, 0] / 255.0) * 1.1)
    result[:, :, 2] = result[:, :, 2] - ((avg_b - 128) * (result[:, :, 0] / 255.0) * 1.1)
    return cv2.cvtColor(result, cv2.COLOR_LAB2BGR)

In [6]:
def adaptive_crop_skin(image_rgb):
    """Isolates the clean skin patch, ignoring card patches and dark borders."""
    h, w = image_rgb.shape[:2]
    img_ycrcb = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2YCrCb)
    
    # Skin color detection mask
    lower_skin = np.array([0, 135, 85], dtype=np.uint8)
    upper_skin = np.array([255, 175, 135], dtype=np.uint8)
    mask = cv2.inRange(img_ycrcb, lower_skin, upper_skin)
    
    # Pre-emptive masking to remove known non-skin areas
    mask[int(h*0.80):h, :] = 0  # Cut bottom 20% (calibration card)
    mask[0:int(h*0.05), :] = 0  # Cut top 5% border
    mask[:, 0:int(w*0.05)] = 0  # Cut left 5% border
    mask[:, int(w*0.95):w] = 0  # Cut right 5% border

    # Morphological cleaning & shrinking
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.erode(mask, kernel, iterations=3) # Pulls away from background edges
    
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        # Fallback if detection fails: take center crop
        return image_rgb[int(h*0.2):int(h*0.7), int(w*0.2):int(w*0.8)]
        
    largest_cnt = max(contours, key=cv2.contourArea)
    x, y, w_box, h_box = cv2.boundingRect(largest_cnt)
    
    # Apply 10% inner inset for a 'pure' skin sample
    ix, iy = int(w_box * 0.1), int(h_box * 0.1)
    return image_rgb[y+iy : y+h_box-iy, x+ix : x+w_box-ix]

In [7]:
def prepare_image_multimodal(img_path):
    img = cv2.imread(img_path)
    if img is None: return None
    
    # 1. Calibrate & Crop
    balanced = white_balance(img)
    rgb_img = cv2.cvtColor(balanced, cv2.COLOR_BGR2RGB)
    skin_patch = adaptive_crop_skin(rgb_img)
    
    # 2. Resize and extract HSV
    resized_rgb = cv2.resize(skin_patch, INPUT_SIZE)
    resized_hsv = cv2.cvtColor(resized_rgb, cv2.COLOR_RGB2HSV)
    
    # 3. Create 6-Channel 1D Sequence (Shape: 16384, 6)
    # Stacking R, G, B, H, S, V
    rgb_data = resized_rgb.reshape(-1, 3) / 255.0
    hsv_data = resized_hsv.reshape(-1, 3) / 255.0
    combined_seq = np.concatenate([rgb_data, hsv_data], axis=-1)
    
    return combined_seq

In [8]:
# ── 3. ROBUST DATA PREPARATION (FIXES NaN LOSS) ──
from sklearn.preprocessing import StandardScaler, MinMaxScaler

X_img = []
X_clin = []
Y_raw = []

print("Processing images and clinical metadata...")
for idx, row in df.iterrows():
    path = os.path.join(IMAGE_DIR, row['image_idx'])
    
    # 1. Process Image to 6-channel 1D sequence (16384, 6)
    img_data = prepare_image_multimodal(path) 
    
    if img_data is not None:
        X_img.append(img_data)
        
        # 2. Collect Clinical Metadata: [gestational_age, age, weight]
        clinical_data = [row['gestational_age'], row['age(day)'], row['weight']]
        X_clin.append(clinical_data)
        
        # 3. Collect Regression Target (Exact Bilirubin level)
        Y_raw.append(row['blood(mg/dL)'])
    
    if idx % 200 == 0:
        print(f"Processed {idx}/{len(df)} samples...")

# Convert lists to Numpy Arrays
X_img = np.array(X_img)
X_clin = np.array(X_clin)
Y_raw = np.array(Y_raw).reshape(-1, 1)

# ── 4. SCALING (CRITICAL FOR NaN PREVENTION) ──

# Scale Inputs (Clinical Metadata)
scaler_x = StandardScaler()
X_clin_scaled = scaler_x.fit_transform(X_clin)

# Scale Targets (Bilirubin) -> This prevents the loss from becoming 'inf' or 'nan'
scaler_y = MinMaxScaler()
Y_scaled = scaler_y.fit_transform(Y_raw).flatten()

# ── TRAIN/TEST SPLIT ──
X_i_tr, X_i_ts, X_c_tr, X_c_ts, y_tr, y_ts = train_test_split(
    X_img, X_clin_scaled, Y_scaled, test_size=0.2, random_state=42
)

# Final numerical safety check
X_i_tr = np.nan_to_num(X_i_tr)
X_c_tr = np.nan_to_num(X_c_tr)

print(f"\nData Preparation Complete!")
print(f"Training Image Shape: {X_i_tr.shape}")
print(f"Training Clinical Shape: {X_c_tr.shape}")
print(f"Target Range: {Y_scaled.min()} to {Y_scaled.max()}")

Processing images and clinical metadata...
Processed 0/2235 samples...
Processed 200/2235 samples...
Processed 400/2235 samples...
Processed 600/2235 samples...
Processed 800/2235 samples...
Processed 1000/2235 samples...
Processed 1200/2235 samples...
Processed 1400/2235 samples...
Processed 1600/2235 samples...
Processed 1800/2235 samples...
Processed 2000/2235 samples...
Processed 2200/2235 samples...

Data Preparation Complete!
Training Image Shape: (1788, 16384, 6)
Training Clinical Shape: (1788, 3)
Target Range: 0.0 to 1.0


In [9]:
# def build_multimodal_1dcnn():
#     img_input = layers.Input(shape=(16384, 6), name="skin_sequence_input")
#     x = layers.Conv1D(32, 3, padding='same', activation='relu')(img_input)
#     x = layers.MaxPooling1D(2, padding='same')(x)
#     x = layers.Conv1D(64, 3, padding='same', activation='relu')(x)
#     x = layers.MaxPooling1D(2, padding='same')(x)
#     x = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
#     x = layers.MaxPooling1D(2, padding='same')(x)
#     x = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
#     x = layers.MaxPooling1D(2, padding='same')(x)
    
#     vis_feat = layers.Flatten()(x)
#     vis_feat = layers.Dense(256, activation='relu')(vis_feat)
#     vis_feat = layers.Dropout(0.3)(vis_feat)

#     clin_input = layers.Input(shape=(3,), name="clinical_input")
#     c = layers.Dense(32, activation='relu')(clin_input)
#     clin_feat = layers.Dense(16, activation='relu')(c)

#     merged = layers.Concatenate()([vis_feat, clin_feat])
#     head = layers.Dense(64, activation='relu')(merged)
#     output = layers.Dense(1, activation='linear', name="tsb_prediction")(head)

#     model = models.Model(inputs=[img_input, clin_input], outputs=output)
    
#     # CRITICAL FIX: Lower learning rate and add clipnorm
#     opt = tf.keras.optimizers.Adam(learning_rate=0.0001, clipnorm=1.0)
    
#     model.compile(optimizer=opt, loss='mse', metrics=['mae'])
#     return model

# model = build_multimodal_1dcnn()

def build_multimodal_1dcnn():
    # --- Visual Branch (Your Original 4-Layer Architecture) ---
    img_input = layers.Input(shape=(16384, 6), name="skin_sequence_input")
    
    # Layer 1: 32 filters
    x = layers.Conv1D(32, 3, padding='same')(img_input)
    x = layers.BatchNormalization()(x) # Keeps math stable
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(2, padding='same')(x)
    
    # Layer 2: 64 filters
    x = layers.Conv1D(64, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(2, padding='same')(x)
    
    # Layer 3: 128 filters
    x = layers.Conv1D(128, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(2, padding='same')(x)
    
    # Layer 4: 128 filters
    x = layers.Conv1D(128, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(2, padding='same')(x)
    
    vis_feat = layers.Flatten()(x)
    vis_feat = layers.Dense(256, activation='relu')(vis_feat)
    vis_feat = layers.Dropout(0.3)(vis_feat)

    # --- Clinical Branch ---
    clin_input = layers.Input(shape=(3,), name="clinical_input")
    c = layers.Dense(32, activation='relu')(clin_input)
    clin_feat = layers.Dense(16, activation='relu')(c)

    # --- Fusion & Output ---
    merged = layers.Concatenate()([vis_feat, clin_feat])
    head = layers.Dense(64, activation='relu')(merged)
    output = layers.Dense(1, activation='linear', name="tsb_prediction")(head)

    model = models.Model(inputs=[img_input, clin_input], outputs=output)
    
    # Using a slightly lower learning rate to prevent the "NaN" explosion
    opt = tf.keras.optimizers.Adam(learning_rate=0.0001, clipnorm=1.0)
    
    model.compile(optimizer=opt, loss='mse', metrics=['mae'])
    return model

model = build_multimodal_1dcnn()
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ skin_sequence_input │ (None, 16384, 6)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 16384, 32) │        608 │ skin_sequence_in… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 16384, 32) │        128 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 16384, 32) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 8192, 32)  │          0 │ activation[0][0]  │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 8192, 64)  │      6,208 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 8192, 64)  │        256 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 8192, 64)  │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 4096, 64)  │          0 │ activation_1[0][… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 4096, 128) │     24,704 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 4096, 128) │        512 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 4096, 128) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 2048, 128) │          0 │ activation_2[0][… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 2048, 128) │     49,280 │ max_pooling1d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 2048, 128) │        512 │ conv1d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 2048, 128) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_3     │ (None, 1024, 128) │          0 │ activation_3[0][… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 131072)    │          0 │ max_pooling1d_3[… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 33,655,089 (128.38 MB)

 Trainable params: 33,654,385 (128.38 MB)

 Non-trainable params: 704 (2.75 KB)

In [10]:
# The model is already built in the previous cell
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ skin_sequence_input │ (None, 16384, 6)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 16384, 32) │        608 │ skin_sequence_in… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 16384, 32) │        128 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 16384, 32) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 8192, 32)  │          0 │ activation[0][0]  │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 8192, 64)  │      6,208 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 8192, 64)  │        256 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 8192, 64)  │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 4096, 64)  │          0 │ activation_1[0][… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 4096, 128) │     24,704 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 4096, 128) │        512 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 4096, 128) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 2048, 128) │          0 │ activation_2[0][… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 2048, 128) │     49,280 │ max_pooling1d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 2048, 128) │        512 │ conv1d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 2048, 128) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_3     │ (None, 1024, 128) │          0 │ activation_3[0][… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 131072)    │          0 │ max_pooling1d_3[… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 33,655,089 (128.38 MB)

 Trainable params: 33,654,385 (128.38 MB)

 Non-trainable params: 704 (2.75 KB)

In [11]:
# Check for NaNs in data
print(f"NaNs in Images: {np.isnan(X_i_tr).any()}")
print(f"NaNs in Clinical: {np.isnan(X_c_tr).any()}")
print(f"NaNs in Labels: {np.isnan(y_tr).any()}")

# If NaNs exist in Labels (Age or Weight might have missing values in CSV)
# We fill them with the column mean
if np.isnan(X_c_tr).any():
    print("Fixing missing clinical values...")
    col_mean = np.nanmean(X_c_tr, axis=0)
    inds = np.where(np.isnan(X_c_tr))
    X_c_tr[inds] = np.take(col_mean, inds[1])

NaNs in Images: False
NaNs in Clinical: False
NaNs in Labels: False


In [12]:
# # ── 5. MULTIMODAL TRAINING ──
# history = model.fit(
#     x={'skin_sequence_input': X_i_tr, 'clinical_input': X_c_tr},
#     y=y_tr,
#     validation_data=({'skin_sequence_input': X_i_ts, 'clinical_input': X_c_ts}, y_ts),
#     epochs=150,
#     batch_size=32,
#     callbacks=[tf.keras.callbacks.TerminateOnNaN()]
# )

history = model.fit(
    x={'skin_sequence_input': X_i_tr, 'clinical_input': X_c_tr},
    y=y_tr,
    validation_data=({'skin_sequence_input': X_i_ts, 'clinical_input': X_c_ts}, y_ts),
    epochs=100,
    batch_size=16
)

Epoch 1/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 24s 196ms/step - loss: 6.1731 - mae: 1.2430 - val_loss: nan - val_mae: nan
Epoch 2/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 22s 195ms/step - loss: 0.0902 - mae: 0.2164 - val_loss: nan - val_mae: nan
Epoch 3/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 22s 197ms/step - loss: 0.0453 - mae: 0.1645 - val_loss: nan - val_mae: nan
Epoch 4/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 22s 200ms/step - loss: 0.0341 - mae: 0.1416 - val_loss: nan - val_mae: nan
Epoch 5/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 22s 201ms/step - loss: 0.0308 - mae: 0.1336 - val_loss: nan - val_mae: nan
Epoch 6/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 23s 202ms/step - loss: 0.0285 - mae: 0.1327 - val_loss: nan - val_mae: nan
Epoch 7/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 23s 203ms/step - loss: 0.0275 - mae: 0.1274 - val_loss: nan - val_mae: nan
Epoch 8/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 23s 204ms/step - loss: 0.0263 - mae: 0.1251 - val_loss: nan - val_mae: nan
Epoch 9/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 23s 205ms/step - loss: 0.0291 -

In [13]:
# ── 6. EVALUATION ──
print("\nEvaluating on Test Set...")
results = model.evaluate(
    x={'skin_sequence_input': X_i_ts, 'clinical_input': X_c_ts},
    y=y_ts
)

print(f"Test MSE: {results[0]:.4f}")
print(f"Test MAE: {results[1]:.4f}")


Evaluating on Test Set...
14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 112ms/step - loss: nan - mae: nan   
Test MSE: nan
Test MAE: nan


**IGNORE**

In [14]:
# Predict on test set
preds_scaled = model.predict({'skin_sequence_input': X_i_ts, 'clinical_input': X_c_ts})

# Convert back to actual mg/dL
preds_real = scaler_y.inverse_transform(preds_scaled)
y_real = scaler_y.inverse_transform(y_ts.reshape(-1, 1))

# Final Error Calculation
mae = np.mean(np.abs(preds_real - y_real))
print(f"Final Prediction Error: {mae:.2f} mg/dL")

14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 95ms/step
Final Prediction Error: nan mg/dL


In [15]:
# ── VERIFICATION: WHAT DOES THE AI SEE? ──
import matplotlib.pyplot as plt

# Pick a random row that requires Treatment (Value 1) to test the hardest cases
test_row = df[df['Treatment'] == 1].iloc[0] 
sample_path = os.path.join(IMAGE_DIR, test_row['image_idx'])

# Process the image through your pipeline
rgb_seq, hsv_seq = prepare_image(sample_path)

# Reconstruct the 1D data (3, 16384) back into a 2D image (128, 128, 3)
reconstructed_rgb = rgb_seq.T.reshape(128, 128, 3)

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB))
plt.title("Original Image (From IMAGE_DIR)")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(reconstructed_rgb)
plt.title("What the AI Sees (Processed Input)")
plt.axis('off')

plt.tight_layout()
plt.show()

print(f"Target Bilirubin: {test_row['blood(mg/dL)']} | Target Treatment: {test_row['Treatment']}")

NameError: name 'prepare_image' is not defined

In [ ]:
# history = model.fit(
#     x={'skin_sequence_input': X_i_tr, 'clinical_input': X_c_tr},
#     y=y_tr,
#     validation_data=({'skin_sequence_input': X_i_ts, 'clinical_input': X_c_ts}, y_ts),
#     epochs=100,
#     batch_size=16
# )

Starting re-tuned training with synchronized dictionaries...
Epoch 1/100


ValueError: Attr 'Toutput_types' of 'OptionalFromValue' Op passed list of length 0 less than minimum 1.

In [ ]:
# # ── 6. EVALUATION ──
# results = model.evaluate(
#     {'rgb_input': X_rgb_test, 'hsv_input': X_hsv_test},
#     {'regression_output': y_reg_test, 'treatment_output': y_tr_test}
# )
# print(f"Final Classification Accuracy: {results[4]*100:.2f}%")

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 23.9164 - regression_output_loss: 23.4687 - regression_output_mae: 3.7598 - treatment_output_accuracy: 0.7673 - treatment_output_loss: 0.4479 
Final Classification Accuracy: 76.73%
